### Silver to Gold: Building BI Ready Tables

### Import helpers used to derive Gold-layer fact metrics.

In [0]:
# Import the dependency used by the lines below.
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType

### Load configurable names and define a simple validation helper for Gold outputs.

In [0]:
# Load the catalog name from Spark config, or use the default project catalog.
catalog_name = spark.conf.get("training_0002_ecommerce.catalog_name", "training_0002_ecommerce")
reference_schema = spark.conf.get("training_0002_ecommerce.reference_schema", "reference")

# Fail early when an expected curated dataset is empty.
# Define a helper that stops the notebook when a required dataset is empty.
def validate_non_empty(df, dataset_name):
    # Count the rows so the dataset can be validated before writing it out.
    row_count = df.count()
    # Check whether the validation condition is met before continuing.
    if row_count == 0:
        # Stop execution with a clear error message when the validation fails.
        raise ValueError(f"{dataset_name} is empty. Validate upstream Silver tables before writing Gold outputs.")
    # Print a small status message so the notebook run is easier to follow.
    print(f"Validated {dataset_name}: {row_count} rows")

### Load the Silver fact table that feeds the Gold transaction model.

In [0]:
# Load the source table needed for this transformation step.
df = spark.table(f"{catalog_name}.silver.slv_order_items")

# Preview the current result to verify the transformation output.
df.limit(10).display()

### Derive transaction-level revenue, discount, coupon, and date metrics.

In [0]:
# 1) Add gross amount
df = df.withColumn(
    "gross_amount",
    F.col("quantity") * F.col("unit_price")
    )

# 2) Add discount_amount (discount_pct is already numeric, e.g., 21 -> 21%)
df = df.withColumn(
    "discount_amount",
    F.ceil(F.col("gross_amount") * (F.col("discount_pct") / 100.0))
)

# 3) Add sale_amount = gross - discount
df = df.withColumn(
    "sale_amount",
    F.col("gross_amount") - F.col("discount_amount") + F.col("tax_amount")
)

# add date id
df = df.withColumn("date_id", F.date_format(F.col("dt"), "yyyyMMdd").cast(IntegerType()))  # Create date_key

# Coupon flag
#  coupon flag = 1 if coupon_code is not null else 0
df = df.withColumn(
    "coupon_flag",
    F.when(F.col("coupon_code").isNotNull(), F.lit(1))
     .otherwise(F.lit(0))
)

# Preview the current result to verify the transformation output.
df.limit(5).display()    

currency conversion

### Define FX rates and persist them as a reusable managed reference table.

In [0]:
# --- 1) Define your fixed FX rates (as of 2025-10-15, like your PBI note) ---
# Define the fixed FX rates used to normalize sales into INR.
fx_rates = {
    "INR": 1.00,
    "AED": 24.18,
    "AUD": 57.55,
    "CAD": 62.93,
    "GBP": 117.98,
    "SGD": 68.18,
    "USD": 88.29,
}

rates = [(k, float(v)) for k, v in fx_rates.items()]
# Convert the FX rate mapping into a Spark DataFrame for joins and persistence.
rates_df = spark.createDataFrame(rates, ["currency", "inr_rate"])
rates_df.show()

In [0]:
# Join each transaction to the FX lookup and calculate INR-normalized sales.
# Start a chained transformation that enriches the dataset step by step.
df = (
    df
    .join(
        rates_df,
        rates_df.currency == F.upper(F.trim(F.col("unit_price_currency"))),
        "left"
    )
    .withColumn("sale_amount_inr", F.col("sale_amount") * F.col("inr_rate"))
    .withColumn("sale_amount_inr", F.ceil(F.col("sale_amount_inr")))
)

In [0]:
# Preview the current result to verify the transformation output.
df.limit(5).display()    

### Select the final fact-table columns and aliases used for BI reporting.

In [0]:
# Select and rename the final columns for the Gold fact table.
orders_gold_df = df.select(
    F.col("date_id"),
    F.col("dt").alias("transaction_date"),
    F.col("order_ts").alias("transaction_ts"),
    F.col("order_id").alias("transaction_id"),
    F.col("customer_id"),
    F.col("item_seq").alias("seq_no"),
    F.col("product_id"),
    F.col("channel"),
    F.col("coupon_code"),
    F.col("coupon_flag"),
    F.col("unit_price_currency"),
    F.col("quantity"),
    F.col("unit_price"),
    F.col("gross_amount"),
    F.col("discount_pct").alias("discount_percent"),
    F.col("discount_amount"),
    F.col("tax_amount"),
    F.col("sale_amount").alias("net_amount"),
    F.col("sale_amount_inr").alias("net_amount_inr")
)

In [0]:
# Preview the current result to verify the transformation output.
orders_gold_df.limit(5).display()

### Persist the final Gold fact table.

In [0]:
# Write raw data to the gold layer (catalog: training_0002_ecommerce, schema: gold, table: gld_fact_order_items)
# Write the current DataFrame to a Delta table in the target layer.
orders_gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.gold.gld_fact_order_items")

Sanity Check

In [0]:
# Run the Spark SQL statement needed for this setup or transformation step.
spark.sql(f"SELECT count(*) FROM {catalog_name}.gold.gld_fact_order_items").show()

In [0]:
fx_rates_table = f"{catalog_name}.{reference_schema}.ref_fx_rates"
validate_non_empty(rates_df, "fx rates reference")
# Write the current DataFrame to a Delta table in the target layer.
rates_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(fx_rates_table)

validate_non_empty(spark.table(fx_rates_table), "ref_fx_rates")
validate_non_empty(spark.table(f"{catalog_name}.gold.gld_fact_order_items"), "gld_fact_order_items")